# Lesson 10: Calling APIs

## Why Learn API Calling?

In Lesson 9, your agent used DataForSEO to search the web. But how does that actually work?

Under the hood, the DataForSEO tool **calls an API** — it sends a request over the internet and gets data back. Every tool your agents use works this way:
- Search tools call search APIs
- Image tools call image APIs
- Storage tools call file system APIs

In this lesson, you'll learn to call APIs yourself. This is how you'll build **custom tools** for your agents later.

> **Cost:** This lesson uses free APIs for most examples. The final exercise uses DataForSEO (optional, requires API key). If you don't have a DataForSEO key, you can still follow along — the concepts are the same.

In [ ]:
# Install httpx — a modern HTTP library for Python
!python -m pip install httpx python-dotenv

## What's an API?

**API** (Application Programming Interface) is how programs talk to each other over the internet.

Think of it like ordering food at a restaurant:
- **You** (your code) send an **order** (request) to the **kitchen** (server)
- The kitchen prepares your food and sends it back as a **response**
- You don't need to know how the kitchen works — you just need to know the **menu** (API docs)

When you call an API, you:
1. Send a **request** to a URL (the API endpoint)
2. Get back a **response** with data (usually in JSON format)

## HTTP Basics: GET vs POST

There are two main types of requests:

| Method | When to Use | Example |
|--------|------------|----------|
| **GET** | Fetching data | "Give me the weather for Hanoi" |
| **POST** | Sending data | "Here's a keyword, analyze it" |

Every request also has:
- **URL** — Where to send it (e.g., `https://api.example.com/search`)
- **Headers** — Metadata like authentication tokens
- **Body** — The data you're sending (POST only)
- **Status code** — The response tells you if it worked:
  - `200` = Success
  - `401` = Unauthorized (bad API key)
  - `404` = Not found
  - `429` = Too many requests (rate limited)
  - `500` = Server error

## Making Requests with `httpx`

Python has several libraries for HTTP requests. We use **httpx** because it's modern, fast, and already in our project's dependencies.

Let's start with a simple GET request to a free, public API.

In [ ]:
import httpx

# GET request to a free public API
response = httpx.get("https://httpbin.org/get")

print(f"Status code: {response.status_code}")
print(f"Content type: {response.headers['content-type']}")
print(f"Response body (first 300 chars):")
print(response.text[:300])

In [ ]:
# POST request — sending data to the server
response = httpx.post(
    "https://httpbin.org/post",
    json={"keyword": "on-page SEO", "location": "United States"},
)

print(f"Status code: {response.status_code}")

# The response shows what the server received
data = response.json()
print(f"Server received our data: {data['json']}")

## Parsing JSON Responses

Most APIs return data in **JSON** format — which maps directly to Python dicts and lists.

The pattern is always:
```python
response = httpx.get("https://api.example.com/data")
data = response.json()  # Converts JSON string → Python dict
value = data["key"]     # Extract what you need
```

Let's try with a real API that returns useful data.

In [ ]:
# Call a free API that returns random user data
response = httpx.get("https://randomuser.me/api/?results=3")
data = response.json()

# Extract just what we need from the nested JSON
for user in data["results"]:
    name = f"{user['name']['first']} {user['name']['last']}"
    country = user["location"]["country"]
    email = user["email"]
    print(f"{name} from {country} — {email}")

## API Keys and Authentication

Most useful APIs require **authentication** — they need to know who's calling.

Common methods:
1. **API key in header** — Most common: `headers={"Authorization": "Bearer YOUR_KEY"}`
2. **Basic auth** — Username + password: `auth=("login", "password")`
3. **API key in URL** — Less secure: `?api_key=YOUR_KEY`

**Important:** Never put API keys directly in your code. Use environment variables:

```python
import os
api_key = os.getenv("MY_API_KEY")  # Reads from .env file
```

In [ ]:
# Example: API key in header
response = httpx.get(
    "https://httpbin.org/headers",
    headers={"Authorization": "Bearer my-secret-key-123"},
)
print("Server saw these headers:")
print(response.json()["headers"])

print()

# Example: Basic auth (username + password)
response = httpx.get(
    "https://httpbin.org/basic-auth/myuser/mypass",
    auth=("myuser", "mypass"),
)
print(f"Basic auth result: {response.json()}")

## Error Handling

APIs can fail for many reasons: bad keys, network issues, rate limits. Always handle errors:

```python
try:
    response = httpx.get(url, timeout=15)
    response.raise_for_status()  # Raises exception for 4xx/5xx status codes
    data = response.json()
except httpx.HTTPStatusError as e:
    print(f"API returned error: {e.response.status_code}")
except httpx.RequestError as e:
    print(f"Network error: {e}")
```

Key points:
- Always set a **timeout** so your code doesn't hang forever
- Use `raise_for_status()` to catch error responses
- Handle different error types separately

In [ ]:
# Handling errors properly

def safe_api_call(url):
    """Make an API call with proper error handling."""
    try:
        response = httpx.get(url, timeout=15)
        response.raise_for_status()
        return response.json()
    except httpx.HTTPStatusError as e:
        print(f"API error {e.response.status_code}: {e.response.text[:100]}")
        return None
    except httpx.RequestError as e:
        print(f"Network error: {e}")
        return None

# This will work
print("Good URL:")
result = safe_api_call("https://httpbin.org/get")
print(f"  Got data: {bool(result)}")

# This will return a 404
print("\nBad URL (404):")
result = safe_api_call("https://httpbin.org/status/404")
print(f"  Got data: {bool(result)}")

## Putting It Together: A Real API Call

Let's combine everything into a function that calls the **DataForSEO** SERP API to check what Google's **AI Overview** says about a keyword.

This is exactly what our product does — the AIO analysis tool calls this same API.

> **Note:** This requires a `DATA_FOR_SEO_API_KEY` in your `.env` file. If you don't have one, read the code and output — the concepts are the same regardless of which API you call.

In [ ]:
import os
import base64
from dotenv import load_dotenv

load_dotenv()


def get_ai_overview(keyword):
    """Check what Google's AI Overview says about a keyword."""
    # Step 1: Get credentials from environment
    raw_key = os.getenv("DATA_FOR_SEO_API_KEY", "")
    if not raw_key:
        print("DATA_FOR_SEO_API_KEY not set — skipping live API call.")
        return None

    # Step 2: Decode the Basic auth credentials
    if raw_key.startswith("Basic "):
        raw_key = raw_key[len("Basic "):]
    decoded = base64.b64decode(raw_key).decode("utf-8")
    login, password = decoded.split(":", 1)

    # Step 3: Make the POST request
    try:
        response = httpx.post(
            "https://api.dataforseo.com/v3/serp/google/organic/live/advanced",
            auth=(login, password),
            json=[{
                "keyword": keyword,
                "location_code": 2840,       # United States
                "language_code": "en",
                "load_async_ai_overview": True,
            }],
            timeout=60,
        )
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        print(f"API call failed: {e}")
        return None

    # Step 4: Parse the response — find the AI Overview item
    tasks = data.get("tasks", [])
    if not tasks or not tasks[0].get("result"):
        return {"keyword": keyword, "has_aio": False}

    items = tasks[0]["result"][0].get("items", [])
    for item in items:
        if item.get("type") == "ai_overview":
            # Extract text from the AI Overview
            sections = []
            references = []
            for element in item.get("items", []):
                if element.get("text"):
                    sections.append(element["text"])
                for ref in element.get("references", []):
                    references.append(ref.get("url", ""))

            return {
                "keyword": keyword,
                "has_aio": True,
                "content": "\n\n".join(sections),
                "sources_cited": references,
            }

    return {"keyword": keyword, "has_aio": False}


# Try it!
result = get_ai_overview("on-page SEO best practices")
if result:
    print(f"Keyword: {result['keyword']}")
    print(f"Has AI Overview: {result['has_aio']}")
    if result['has_aio']:
        print(f"\nAI Overview content (first 500 chars):")
        print(result['content'][:500])
        print(f"\nSources cited: {len(result['sources_cited'])}")
        for url in result['sources_cited'][:5]:
            print(f"  - {url}")

## Connection to Agents

Everything you just learned is what happens inside agent tools:

1. **DataForSEOSearchTools** calls the DataForSEO SERP API (POST with Basic auth) — used by Content Writer
2. **DataForSEOImageTools** calls the DataForSEO Images API (POST with Basic auth) — used by Image Finder
3. **AIOTools** calls the DataForSEO AI Overview API (POST with Basic auth) — used by AIO Analyzer

When you build a custom tool for an agent, you're writing a function that:
- Takes parameters from the agent
- Calls an API using `httpx`
- Returns the parsed result

In the next lesson, you'll learn to make agents return **structured data** — specific fields in a predictable format.

## Lesson 10 Summary

What you learned:
- **APIs** are how programs talk to each other over the internet
- **GET** fetches data, **POST** sends data
- **httpx** makes HTTP requests in Python
- **API keys** go in environment variables, never in code
- **Always handle errors** — set timeouts, check status codes
- **JSON parsing** — `response.json()` gives you a Python dict
- Agent tools are just functions that call APIs

**Next lesson:** Structured output — making agents return data in a specific format.

## Exercise

Write a function that calls the [JSONPlaceholder](https://jsonplaceholder.typicode.com) API to:
1. Fetch a list of posts from `https://jsonplaceholder.typicode.com/posts`
2. Filter to only posts by user ID 1
3. Print each post's title

Include proper error handling with try/except.

In [ ]:
# Exercise: Fill in the function

import httpx


def get_user_posts(user_id):
    """Fetch posts by a specific user from JSONPlaceholder."""
    try:
        response = httpx.get(
            ___,          # API URL
            timeout=___,  # Set a timeout
        )
        response.raise_for_status()
        all_posts = response.json()

        # Filter posts by user_id
        user_posts = [p for p in all_posts if p[___] == user_id]
        return user_posts

    except Exception as e:
        print(f"Error: {e}")
        return []


posts = get_user_posts(1)
print(f"Found {len(posts)} posts by user 1:\n")
for post in posts:
    print(f"  - {post['title']}")

<details>
<summary>Click to reveal answer</summary>

```python
import httpx


def get_user_posts(user_id):
    """Fetch posts by a specific user from JSONPlaceholder."""
    try:
        response = httpx.get(
            "https://jsonplaceholder.typicode.com/posts",
            timeout=15,
        )
        response.raise_for_status()
        all_posts = response.json()

        user_posts = [p for p in all_posts if p["userId"] == user_id]
        return user_posts

    except Exception as e:
        print(f"Error: {e}")
        return []


posts = get_user_posts(1)
print(f"Found {len(posts)} posts by user 1:\n")
for post in posts:
    print(f"  - {post['title']}")
```
</details>